In [1]:
import xarray as xr

In [3]:
ds = xr.open_dataset("/home/ddyob/Documents/argo_piloting/sim/data/argo_data/argo_4903784/4903784_profiles.nc")

In [ ]:
ds

In [5]:

import numpy as np
import plotly.graph_objects as go

# Sort profiles by time
idx = np.argsort(ds['TIME'].values)
lat  = ds['LATITUDE'].values[idx]
lon  = ds['LONGITUDE'].values[idx]
time = ds['TIME'].values[idx]
pres = ds['PRES'].values[idx]          # (N_PROF, N_LEVELS)

time_num = (time - time[0]).astype('float64') / 1e9 / 86400  # days since first profile

# --- build vertical profile traces ---
traces = []
for i in range(len(idx)):
    valid = np.isfinite(pres[i])
    if not valid.any():
        continue
    depth = -pres[i, valid]            # pressure (dbar) ≈ depth (m), negative = below surface
    n     = valid.sum()
    traces.append(go.Scatter3d(
        x=np.full(n, lon[i]),
        y=np.full(n, lat[i]),
        z=depth,
        mode='lines',
        line=dict(
            color=np.full(n, time_num[i]),
            colorscale='Viridis',
            cmin=time_num.min(),
            cmax=time_num.max(),
            width=2,
        ),
        showlegend=False,
        hovertemplate=(
            f"Cycle {i+1}<br>"
            f"Lon: {lon[i]:.3f}°<br>"
            f"Lat: {lat[i]:.3f}°<br>"
            "Depth: %{z:.0f} m<extra></extra>"
        ),
    ))

# --- surface track ---
traces.append(go.Scatter3d(
    x=lon, y=lat, z=np.zeros(len(lon)),
    mode='lines+markers',
    name='Surface track',
    line=dict(color='white', width=2),
    marker=dict(
        size=3,
        color=time_num,
        colorscale='Viridis',
        cmin=time_num.min(),
        cmax=time_num.max(),
        colorbar=dict(title='Days since<br>first profile', x=1.02),
    ),
    hovertemplate="Lon: %{x:.3f}°<br>Lat: %{y:.3f}°<extra>Surface</extra>",
))

# --- waterline plane ---
lon_grid = np.linspace(lon.min(), lon.max(), 4)
lat_grid = np.linspace(lat.min(), lat.max(), 4)
LON, LAT = np.meshgrid(lon_grid, lat_grid)
traces.append(go.Surface(
    x=LON, y=LAT, z=np.zeros_like(LON),
    colorscale=[[0, 'rgba(0,120,200,0.25)'], [1, 'rgba(0,120,200,0.25)']],
    showscale=False,
    name='Waterline',
    hoverinfo='skip',
))

fig = go.Figure(data=traces)
fig.update_layout(
    title='Argo Float 4903784 — 3D Profile Track',
    scene=dict(
        xaxis_title='Longitude (°)',
        yaxis_title='Latitude (°)',
        zaxis_title='Depth (m)',
        bgcolor='rgb(10, 30, 60)',
        xaxis=dict(gridcolor='grey'),
        yaxis=dict(gridcolor='grey'),
        zaxis=dict(gridcolor='grey', autorange='reversed'),
    ),
    paper_bgcolor='rgb(15, 20, 40)',
    font_color='white',
    margin=dict(l=0, r=0, t=40, b=0),
    height=700,
)
fig.show()


## Get command
Okay, so the goal of this is to get the commands sent to the float

In [11]:
# Sort profiles by time
from matplotlib import pyplot as plt
idx = np.argsort(ds['TIME'].values)
lat  = ds['LATITUDE'].values[idx]
lon  = ds['LONGITUDE'].values[idx]
time = ds['TIME'].values[idx]
pres = ds['PRES'].values[idx]
ds

<xarray.Dataset> Size: 2MB
Dimensions:          (N_PROF: 376, N_LEVELS: 394)
Coordinates:
  * N_PROF           (N_PROF) int64 3kB 188 189 187 190 186 ... 373 1 374 0 375
    LATITUDE         (N_PROF) float64 3kB 55.06 55.07 55.07 ... 55.13 55.14
    LONGITUDE        (N_PROF) float64 3kB 14.33 14.33 14.33 ... 14.51 14.51
    TIME             (N_PROF) datetime64[ns] 3kB 2023-11-07T20:40:05 ... 2025...
  * N_LEVELS         (N_LEVELS) int64 3kB 0 1 2 3 4 5 ... 389 390 391 392 393
Data variables: (12/15)
    CYCLE_NUMBER     (N_PROF) int64 3kB 1 1 2 2 3 3 ... 188 188 189 189 190 190
    DATA_MODE        (N_PROF) <U1 2kB 'R' 'R' 'R' 'R' 'R' ... 'R' 'R' 'R' 'R'
    DIRECTION        (N_PROF) <U1 2kB 'D' 'A' 'D' 'A' 'D' ... 'D' 'A' 'D' 'A'
    PLATFORM_NUMBER  (N_PROF) int64 3kB 4903784 4903784 ... 4903784 4903784
    POSITION_QC      (N_PROF) int64 3kB 1 1 1 1 1 1 1 1 1 ... 1 1 1 1 1 1 1 1 1
    PRES             (N_PROF, N_LEVELS) float32 593kB 0.49 0.66 0.79 ... nan nan
    ...               ...
    PSAL_ERROR       (N_PROF) float32 2kB nan nan nan nan ... nan nan nan nan
    PSAL_QC          (N_PROF) int64 3kB 1 1 1 1 1 1 1 1 1 ... 1 1 1 1 1 1 1 1 1
    TEMP             (N_PROF, N_LEVELS) float32 593kB 9.573 9.573 ... nan nan
    TEMP_ERROR       (N_PROF) float32 2kB nan nan nan nan ... nan nan nan nan
    TEMP_QC          (N_PROF) int64 3kB 1 1 1 1 1 1 1 1 1 ... 1 1 1 1 1 1 1 1 1
    TIME_QC          (N_PROF) int64 3kB 1 1 1 1 1 1 1 1 1 ... 1 1 1 1 1 1 1 1 1
Attributes:
    DATA_ID:              ARGO
    DOI:                  http://doi.org/10.17882/42182
    Fetched_from:         erddap.ifremer.fr
    Fetched_by:           ddyob
    Fetched_date:         2026/03/25
    Fetched_constraints:  WMO4903784
    Fetched_uri:          https://erddap.ifremer.fr/erddap/tabledap/ArgoFloat...
    Processing_history:   [PRES,TEMP,PSAL] real-time and adjusted/delayed var...

In [17]:
trajectory = xr.open_dataset("~/Documents/argo_piloting/sim/data/argo_data/argo_4903784/4903784_Rtraj.nc")

/tmp/ipykernel_107809/3427450809.py:1: FutureWarning: In a future version, xarray will not decode the variable 'CLOCK_OFFSET' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  trajectory = xr.open_dataset("~/Documents/argo_piloting/sim/data/argo_data/argo_4903784/4903784_Rtraj.nc")


In [ ]:
trajectory

In [21]:
# See all variables
for v in trajectory.data_vars:
    print(f"{v}: {trajectory[v].dims}, {trajectory[v].shape}")

DATA_TYPE: (), ()
FORMAT_VERSION: (), ()
HANDBOOK_VERSION: (), ()
REFERENCE_DATE_TIME: (), ()
DATE_CREATION: (), ()
DATE_UPDATE: (), ()
PLATFORM_NUMBER: (), ()
PROJECT_NAME: (), ()
PI_NAME: (), ()
TRAJECTORY_PARAMETERS: ('N_PARAM',), (44,)
DATA_CENTRE: (), ()
DATA_STATE_INDICATOR: (), ()
PLATFORM_TYPE: (), ()
FLOAT_SERIAL_NO: (), ()
FIRMWARE_VERSION: (), ()
WMO_INST_TYPE: (), ()
POSITIONING_SYSTEM: (), ()
JULD: ('N_MEASUREMENT',), (452746,)
JULD_STATUS: ('N_MEASUREMENT',), (452746,)
JULD_QC: ('N_MEASUREMENT',), (452746,)
JULD_ADJUSTED: ('N_MEASUREMENT',), (452746,)
JULD_ADJUSTED_STATUS: ('N_MEASUREMENT',), (452746,)
JULD_ADJUSTED_QC: ('N_MEASUREMENT',), (452746,)
JULD_DATA_MODE: ('N_MEASUREMENT',), (452746,)
LATITUDE: ('N_MEASUREMENT',), (452746,)
LONGITUDE: ('N_MEASUREMENT',), (452746,)
POSITION_ACCURACY: ('N_MEASUREMENT',), (452746,)
POSITION_QC: ('N_MEASUREMENT',), (452746,)
AXES_ERROR_ELLIPSE_MAJOR: ('N_MEASUREMENT',), (452746,)
AXES_ERROR_ELLIPSE_MINOR: ('N_MEASUREMENT',), (452746

In [ ]:
ds = trajectory

# First, check what measurement codes exist
# Descent codes are usually 500-599, ascent 700-799
# Adjust these after checking the output above
DESC_CODE = 190
ASC_CODE = 590

desc_speeds = []
asc_speeds = []

for cyc in np.unique(ds.CYCLE_NUMBER.values):
    if np.isnan(cyc):
        continue
    cmask = ds.CYCLE_NUMBER == cyc

    for label, codes, speeds_list in [("desc", DESC_CODE, desc_speeds),
                                       ("asc", ASC_CODE, asc_speeds)]:
        mask = ds.MEASUREMENT_CODE == codes
        total_mask = cmask & mask
        sub = ds.sel(N_MEASUREMENT=mask)

        if len(sub.N_MEASUREMENT) < 2:
            continue

        pres = sub.PRES.values
        print(pres,juld)
        juld = sub.JULD.values
        dt = (juld[-1] - juld[0]) / np.timedelta64(1, "s")  # total seconds
        dp = pres[-1] - pres[0]  # total pressure change

        if dt > 0:
            speed = (abs(dp)) / (dt)  # dbar/hour
            speeds_list.append((int(cyc), speed, dt / 60))  # cycle, speed, duration_min

# Convert to arrays
desc_arr = np.array(desc_speeds)  # columns: cycle, speed, duration_min
asc_arr = np.array(asc_speeds)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(desc_arr[:, 0], desc_arr[:, 1], ".-", label="Descent")
axes[0].plot(asc_arr[:, 0], asc_arr[:, 1], ".-", label="Ascent")
axes[0].set_ylabel("Speed (dbar/hr)")
axes[0].set_title("Profiling speed by cycle")
axes[0].legend()

axes[1].plot(desc_arr[:, 0], desc_arr[:, 2], ".-", label="Descent")
axes[1].plot(asc_arr[:, 0], asc_arr[:, 2], ".-", label="Ascent")
axes[1].set_ylabel("Duration (min)")
axes[1].set_xlabel("Cycle number")
axes[1].legend()

plt.tight_layout()
plt.show()

[ 0.11  0.    0.   ... 43.   43.   43.  ] ['2023-11-07T22:14:22.000000000' '2023-11-07T22:14:27.000000000'
 '2023-11-07T22:14:27.000000000' ... '2025-07-06T10:34:43.000000000'
 '2025-07-06T10:34:47.000000000' '2025-07-06T10:34:49.000000000']
[34.8  34.7  34.7  ...  0.    0.21  0.05] ['2023-11-07T20:41:20.000000000' '2023-11-07T20:41:24.000000000'
 '2023-11-07T20:41:24.000000000' ... '2025-07-01T12:36:48.000000000'
 '2025-07-01T12:38:48.000000000' '2025-07-01T12:40:48.000000000']
